<a href="https://colab.research.google.com/github/sanjil18/Fine-tuning-project/blob/main/02_finetune_qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — QLoRA Fine-Tuning

Fine-tunes a small instruction model (Qwen2.5-1.5B-Instruct by default) on `data/qa_dataset.jsonl` using QLoRA (4-bit base model + LoRA adapters), on a free Colab T4 GPU.

**Before running:** In Colab, go to `Runtime > Change runtime type > T4 GPU`.

**Upload your dataset:** either mount Google Drive, or use the file upload cell below to upload `qa_dataset.jsonl` directly.

## 1. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets

## 2. Upload your dataset

Run this cell and select `qa_dataset.jsonl` from your computer. Skip this cell if you're mounting Drive instead.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select qa_dataset.jsonl
DATA_PATH = list(uploaded.keys())[0] if uploaded else "qa_dataset.jsonl"
print(f"Using dataset file: {DATA_PATH}")

## 3. Config — model, LoRA, training hyperparameters

All the choices you'll want to explain/ablate in your write-up are collected here.

In [ ]:
# --- Model ---
# Swap to "meta-llama/Llama-3.2-1B-Instruct" if you prefer (needs HF login/token for gated access).
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# --- LoRA config ---
# rank (r): capacity of the adapter. Higher = more expressive, more params, more overfitting risk
#   on a small (50-pair) dataset. r=8 is a safe starting point; try r=16 as an ablation.
LORA_R = 8
LORA_ALPHA = 16          # scaling factor, commonly set to 2x rank
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]  # attention projections

# --- Training ---
# With only 50 examples, a few epochs is enough; more risks overfitting/memorizing exact wording.
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4     # typical for LoRA (much higher than full fine-tuning, since only
                         # a small number of adapter params are being updated)
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4     # effective batch size = BATCH_SIZE * GRAD_ACCUM_STEPS = 8
MAX_SEQ_LENGTH = 512

OUTPUT_DIR = "lora_adapter"

## 4. Load dataset and format as instruction-response prompts

In [ ]:
import json
from datasets import Dataset

records = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Loaded {len(records)} Q&A pairs")
print(records[0])

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(example):
    """
    Formats each {instruction, output} pair using the model's chat template,
    so the model sees the same structure at fine-tuning time as at inference time.
    """
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = Dataset.from_list(records)
dataset = dataset.map(format_example)

# Small held-out split just to sanity-check training isn't wildly overfitting;
# with only 50 examples this is illustrative, not a rigorous validation set.
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)}  |  Eval: {len(eval_dataset)}")
print("\n--- Example formatted training text ---\n")
print(train_dataset[0]["text"])

## 5. Load base model in 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False  # required for gradient checkpointing during training

## 6. Attach LoRA adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # confirm only a small % of params are trainable

## 7. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    bf16=True,
    report_to="none",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

# --- If you hit a CUDA out-of-memory (OOM) error ---
# This is expected/normal on small GPUs, not a sign something is broken. Try, in order:
#   1. Lower BATCH_SIZE to 1 and raise GRAD_ACCUM_STEPS to keep the same effective batch size
#   2. Lower MAX_SEQ_LENGTH (e.g. to 256)
#   3. Restart the Colab runtime to clear GPU memory before retrying

trainer.train()

## 8. Save the LoRA adapter

In [ ]:
FINAL_ADAPTER_DIR = "lora_adapter_final"
trainer.save_model(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print(f"Saved adapter to {FINAL_ADAPTER_DIR}")

# Zip it up for easy download to your project's models/lora_adapter/ folder
import shutil
shutil.make_archive("lora_adapter_final", "zip", FINAL_ADAPTER_DIR)

from google.colab import files
files.download("lora_adapter_final.zip")

## 9. Quick sanity-check inference

Ask the fine-tuned model a couple of questions from your dataset (and one it wasn't trained on) to eyeball whether training actually worked before moving to full evaluation in Week 3.

In [ ]:
def ask(question, max_new_tokens=150):
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Q: {question}\nA: {response}\n")

# A question that IS in the training data (should answer well)
ask(train_dataset[0]["text"].split("\n")[0] if False else records[0]["instruction"])

# A question that is NOT in the training data or notes at all — this is the interesting one
# for your RAG-vs-fine-tuning comparison (Week 3): does it hallucinate confidently, or hedge?
ask("What is the capital of Mongolia?")
ask("Explain the CAP theorem's relationship to microservices architecture in detail.")